# Week 4: Final Evaluation and API Demo

## Objective
Validate Department + Sentiment models and demonstrate production API contract for deployment readiness.

## 1. Imports and Setup

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.evaluation.evaluate_all import evaluate_department_model, evaluate_sentiment_model
from src.models.predict import DepartmentPredictor
from src.models.predict_sentiment import SentimentPredictor
from src.scoring.urgency import calculate_urgency_score

## 2. Run Final Evaluation

In [ ]:
department_metrics = evaluate_department_model(dataset_path=project_root / 'data/interim/cleaned_data.csv')
sentiment_metrics = evaluate_sentiment_model(dataset_path=project_root / 'data/interim/cleaned_data.csv')

summary = {
    'department': {k: v for k, v in department_metrics.items() if k != 'classification_report'},
    'sentiment': {k: v for k, v in sentiment_metrics.items() if k != 'classification_report'},
}
print(json.dumps(summary, indent=2))

Observation: Evaluation script outputs macro-level metrics and writes confusion matrices under `reports/`.

## 3. Artifact Verification

In [ ]:
required_paths = [
    project_root / 'artifacts/models/best_department_model.pkl',
    project_root / 'artifacts/vectorizers/tfidf_vectorizer.pkl',
    project_root / 'artifacts/encoders/department_label_encoder.pkl',
    project_root / 'artifacts/models/best_sentiment_model.pkl',
    project_root / 'artifacts/vectorizers/sentiment_tfidf_vectorizer.pkl',
    project_root / 'artifacts/encoders/sentiment_label_encoder.pkl',
]

for path in required_paths:
    print(f"{path.name}: {'OK' if path.exists() else 'MISSING'}")

## 4. End-to-End Inference (Local, No API Server)

In [ ]:
dept_predictor = DepartmentPredictor()
sent_predictor = SentimentPredictor()

sample_text = 'No water supply for 3 days and nobody is responding'
department = dept_predictor.predict([sample_text])[0]
department_conf = float(dept_predictor.predict_proba([sample_text])[0].max())

sent_result = sent_predictor.predict_one(sample_text)
urgency_score, priority_band = calculate_urgency_score(sentiment=sent_result.sentiment, confidence=sent_result.confidence)

print('department:', department)
print('department_confidence:', round(department_conf, 4))
print('sentiment:', sent_result.sentiment)
print('sentiment_confidence:', round(sent_result.confidence, 4))
print('urgency_score:', urgency_score)
print('priority_band:', priority_band)

## 5. API Contract Demo

In [ ]:
example_request = {
    'text': 'No water supply for 3 days and nobody is responding'
}

example_response_shape = {
    'department': 'Water',
    'department_confidence': 0.0,
    'sentiment': 'Critical/Urgent',
    'sentiment_confidence': 0.0,
    'urgency_score': 0,
    'priority_band': 'CRITICAL',
    'model_version': 'week4-1.0.0',
    'timestamp': 'ISO-8601 UTC'
}

print('POST /predict request:')
print(json.dumps(example_request, indent=2))
print('
Expected response schema:')
print(json.dumps(example_response_shape, indent=2))

## 6. Final Review Checklist

- Department and sentiment models evaluated with confusion matrices.
- All required artifacts saved for reproducible inference.
- API contract validated for deployment handoff.
- Pipeline is ready for final project review and extension to transformer upgrades.